[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/01_Multimodal_Foundations/03_fusion_strategies/03_fusion_strategies.ipynb)

# 03. Fusion Strategies: How to Combine Modalities

**This notebook covers:**
- Early Fusion, Late Fusion, Cross-Modal Fusion — built from scratch
- When to use which strategy
- Cross-Attention: the most powerful fusion mechanism
- Gated fusion and attention-based pooling

---

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/01_Multimodal_Foundations/03_fusion_strategies")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from utils.visualization import *
from utils.helpers import count_parameters

set_style()

In [ ]:
# Visualize all three fusion strategies side-by-side
fig = draw_fusion_comparison()
plt.savefig('../assets/fusion_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 1. Early Fusion

**Idea:** Concatenate raw inputs (or early features) BEFORE the main encoder.  
**When:** When modalities are similar in structure or you want maximum interaction.

### Early Fusion — Mathematical Formulation

In early fusion, tokens from all modalities are concatenated **before** joint processing:

$$h_{\text{early}} = \text{Transformer}([\mathbf{v}_1, \ldots, \mathbf{v}_N, \mathbf{t}_1, \ldots, \mathbf{t}_M])$$

where $\mathbf{v}_i \in \mathbb{R}^d$ are image patch embeddings and $\mathbf{t}_j \in \mathbb{R}^d$ are text token embeddings.

**Computational cost:** Self-attention over the combined sequence has complexity:

$$O\left((N + M)^2 \cdot d\right)$$

This is **quadratic in total sequence length** — if you have 196 image patches + 77 text tokens = 273 tokens, attention computes a $273 \times 273$ matrix at every layer. For long sequences this becomes prohibitively expensive, which is why most modern VLMs prefer cross-attention over early fusion.

In [ ]:
class EarlyFusion(nn.Module):
    """Concatenate image patches and text tokens, then process jointly."""
    def __init__(self, img_dim=128, txt_dim=128, hidden_dim=256, n_classes=10):
        super().__init__()
        self.img_proj = nn.Linear(img_dim, hidden_dim)
        self.txt_proj = nn.Linear(txt_dim, hidden_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=4, dim_feedforward=512, batch_first=True
        )
        self.joint_encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.classifier = nn.Linear(hidden_dim, n_classes)

    def forward(self, img_tokens, txt_tokens):
        # Project both to same dimension
        img = self.img_proj(img_tokens)    # [B, N_img, hidden]
        txt = self.txt_proj(txt_tokens)    # [B, N_txt, hidden]
        
        # EARLY FUSION: concatenate along sequence dimension
        combined = torch.cat([img, txt], dim=1)  # [B, N_img+N_txt, hidden]
        
        # Joint processing (all tokens attend to all tokens)
        output = self.joint_encoder(combined)
        
        # Pool and classify
        pooled = output.mean(dim=1)
        return self.classifier(pooled)


model = EarlyFusion()
img_tokens = torch.randn(2, 16, 128)   # 16 image patches
txt_tokens = torch.randn(2, 8, 128)    # 8 text tokens
out = model(img_tokens, txt_tokens)
print(f"Early Fusion output: {out.shape}")
count_parameters(model)

## 2. Late Fusion

**Idea:** Process each modality independently → combine only the final representations.  
**When:** Modalities are very different, or you want to reuse pretrained encoders.

### Late Fusion — Mathematical Forms

Each modality is encoded independently, then combined at the representation level:

| Operation | Formula | Parameters |
|-----------|---------|------------|
| **Concat** | $h = W [f_v; f_t] + b$ | $O((d_v + d_t) \cdot d_{\text{out}})$ |
| **Add** | $h = f_v + f_t$ | $O(1)$ — requires $d_v = d_t$ |
| **Multiply (Hadamard)** | $h = f_v \odot f_t$ | $O(1)$ — element-wise gating |

**Complexity advantage:** Late fusion costs $O(N^2 d + M^2 d)$ — each modality's encoder runs independently with its own self-attention. There is **no cross-modal attention** until the final combination step, making it cheap but limiting interaction depth.

In [ ]:
class LateFusion(nn.Module):
    """Process modalities separately, combine at the end."""
    def __init__(self, img_dim=128, txt_dim=128, hidden_dim=256, 
                 n_classes=10, fusion_type='concat'):
        super().__init__()
        self.fusion_type = fusion_type

        # Separate encoders (in practice these are pretrained)
        self.img_encoder = nn.Sequential(
            nn.Linear(img_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.txt_encoder = nn.Sequential(
            nn.Linear(txt_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        if fusion_type == 'concat':
            self.classifier = nn.Linear(hidden_dim * 2, n_classes)
        else:  # add or multiply
            self.classifier = nn.Linear(hidden_dim, n_classes)

    def forward(self, img_feat, txt_feat):
        img = self.img_encoder(img_feat)   # [B, hidden]
        txt = self.txt_encoder(txt_feat)   # [B, hidden]

        if self.fusion_type == 'concat':
            fused = torch.cat([img, txt], dim=-1)
        elif self.fusion_type == 'add':
            fused = img + txt
        elif self.fusion_type == 'multiply':
            fused = img * txt

        return self.classifier(fused)


# Compare all three late fusion types
img_feat = torch.randn(2, 128)
txt_feat = torch.randn(2, 128)

for ftype in ['concat', 'add', 'multiply']:
    model = LateFusion(fusion_type=ftype)
    out = model(img_feat, txt_feat)
    params = sum(p.numel() for p in model.parameters())
    print(f"Late Fusion ({ftype:8s}): output={out.shape}, params={params:,}")

## 3. Cross-Modal Attention Fusion (Most Powerful)

**Idea:** Let one modality attend to the other using cross-attention.  
This is what you already know! Q from one modality, K/V from another.

### Cross-Attention Formula

Cross-attention lets one modality **query** another. Text tokens ask: *"Which image patches are relevant to me?"*

$$\text{CrossAttn}(T, I) = \text{softmax}\left(\frac{(T W^Q)(I W^K)^\top}{\sqrt{d_k}}\right)(I W^V)$$

| Role | Source | Meaning |
|------|--------|---------|
| **Query (Q)** | Text tokens | "What am I looking for?" |
| **Key (K)** | Image patches | "What do I contain?" |
| **Value (V)** | Image patches | "What information do I provide?" |

Each text token learns **which image patches are relevant** — the word "cat" should attend strongly to patches containing the cat. This is the core mechanism in **LLaVA**, **Flamingo**, and **BLIP-2**, where a frozen vision encoder's patch tokens are cross-attended by an LLM's text tokens.

In [ ]:
class CrossModalFusion(nn.Module):
    """Cross-attention between image and text representations."""
    def __init__(self, dim=128, n_heads=4, n_layers=2, n_classes=10):
        super().__init__()
        
        # Text attends to image (text queries, image keys/values)
        self.cross_attn_layers = nn.ModuleList([
            nn.MultiheadAttention(dim, n_heads, batch_first=True)
            for _ in range(n_layers)
        ])
        self.norms = nn.ModuleList([
            nn.LayerNorm(dim) for _ in range(n_layers)
        ])
        self.ffns = nn.ModuleList([
            nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))
            for _ in range(n_layers)
        ])
        self.ffn_norms = nn.ModuleList([
            nn.LayerNorm(dim) for _ in range(n_layers)
        ])
        
        self.classifier = nn.Linear(dim, n_classes)
        self._attn_weights = []

    def forward(self, txt_tokens, img_tokens, return_attention=False):
        self._attn_weights = []
        x = txt_tokens
        
        for cross_attn, norm, ffn, ffn_norm in zip(
            self.cross_attn_layers, self.norms, self.ffns, self.ffn_norms
        ):
            # Cross-attention: text (Q) attends to image (K, V)
            attn_out, attn_w = cross_attn(x, img_tokens, img_tokens)
            self._attn_weights.append(attn_w.detach())
            x = norm(x + attn_out)
            x = ffn_norm(x + ffn(x))
        
        pooled = x.mean(dim=1)
        logits = self.classifier(pooled)
        
        if return_attention:
            return logits, self._attn_weights
        return logits


model = CrossModalFusion(dim=128, n_heads=4, n_layers=2)
txt_tokens = torch.randn(1, 8, 128)    # 8 text tokens
img_tokens = torch.randn(1, 16, 128)   # 16 image patches

logits, attn_weights = model(txt_tokens, img_tokens, return_attention=True)
print(f"Output: {logits.shape}")
print(f"Attention weights: {len(attn_weights)} layers, each {attn_weights[0].shape}")

count_parameters(model)

In [ ]:
# Visualize cross-attention weights
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Cross-Attention: Text → Image', fontsize=16, fontweight='bold')

txt_labels = ['[CLS]', 'a', 'cat', 'on', 'a', 'sofa', '.', '[SEP]']
img_labels = [f'patch_{i}' for i in range(16)]

for i, (ax, weights) in enumerate(zip(axes, attn_weights)):
    w = weights[0].numpy()  # [txt_len, img_len]
    im = ax.imshow(w, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(16))
    ax.set_xticklabels(img_labels, rotation=45, fontsize=8)
    ax.set_yticks(range(8))
    ax.set_yticklabels(txt_labels)
    ax.set_title(f'Layer {i+1} Cross-Attention', fontsize=12)
    ax.set_xlabel('Image Patches (Keys)')
    ax.set_ylabel('Text Tokens (Queries)')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.savefig('../assets/cross_attention_weights.png', dpi=150, bbox_inches='tight')
plt.show()
print("Each row shows which image patches a text token pays attention to.")
print("After training, 'cat' should attend to the patches containing the cat!")

## 4. Gated Fusion (Adaptive Weighting)

**Idea:** Learn to weight how much each modality contributes.

### Gated Fusion — The Math

The gate learns an **adaptive weight** for each sample based on both modalities:

$$g = \sigma(W_g [f_v; f_t] + b_g)$$

where $\sigma$ is the sigmoid function ($g \in [0, 1]$), and the fused representation is:

$$h = g \odot f_v + (1 - g) \odot f_t$$

**Why this matters:**
- When the image is **noisy or missing**, the gate can shift toward text ($g \to 0$)
- When text is **ambiguous**, the gate can rely on vision ($g \to 1$)
- Unlike fixed-weight late fusion (add/multiply), gating is **input-dependent** — each sample gets its own modality balance

This is especially useful in real-world settings where one modality may be corrupted (blurry image, OCR errors in text).

In [ ]:
class GatedFusion(nn.Module):
    """Learned gating to control modality contribution."""
    def __init__(self, dim=128, n_classes=10):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.ReLU(),
            nn.Linear(dim, 1),
            nn.Sigmoid()  # gate value between 0 and 1
        )
        self.classifier = nn.Linear(dim, n_classes)

    def forward(self, img_feat, txt_feat):
        combined = torch.cat([img_feat, txt_feat], dim=-1)
        gate_value = self.gate(combined)  # [B, 1] between 0 and 1
        
        # Weighted combination
        fused = gate_value * img_feat + (1 - gate_value) * txt_feat
        return self.classifier(fused), gate_value


model = GatedFusion()
img_feat = torch.randn(8, 128)
txt_feat = torch.randn(8, 128)

logits, gates = model(img_feat, txt_feat)

# Visualize gate values
fig, ax = plt.subplots(figsize=(8, 4))
gate_vals = gates.detach().numpy().flatten()
bars = ax.bar(range(len(gate_vals)), gate_vals, color='#9B59B6', alpha=0.7)
ax.axhline(y=0.5, color='red', linestyle='--', label='Equal weighting')
ax.set_xlabel('Sample')
ax.set_ylabel('Gate Value')
ax.set_title('Gated Fusion: Image Contribution Weight\n(>0.5 = more image, <0.5 = more text)', 
             fontsize=12, fontweight='bold')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

### Bilinear Fusion

Beyond additive and multiplicative combinations, **bilinear fusion** captures *multiplicative interactions* between modalities:

$$h = f_v^\top W f_t$$

where $W \in \mathbb{R}^{d_v \times d_t \times d_{\text{out}}}$ is a 3D weight tensor. This lets the model learn that certain visual features *combined with* certain text features produce a strong signal — e.g., "red" + red-patch activation.

**The problem:** Full bilinear fusion requires $d_v \times d_t \times d_{\text{out}}$ parameters. For $d_v = d_t = d_{\text{out}} = 768$, that's **450M parameters** just for the fusion layer!

**Low-rank approximation (MLB — Multimodal Low-rank Bilinear):**

$$h = (U^\top f_v) \odot (V^\top f_t)$$

where $U, V \in \mathbb{R}^{d \times k}$ with rank $k \ll d$. Parameter count drops to **$2 \times d \times k$** (e.g., $k=256$ → ~393K params). This is used in VQA models like **MCB** and **MUTAN** to capture rich cross-modal interactions without the full tensor cost.

## Fusion Strategy Comparison

| Strategy | Pros | Cons | Computational Cost | Best For |
|----------|------|------|--------------------|----------|
| **Early** | Maximum interaction | Expensive, needs same structure | $O((N+M)^2 d)$ — quadratic in joint sequence | Similar modalities |
| **Late** | Simple, reuse encoders | Limited interaction | $O(N^2 d + M^2 d)$ — independent encoders | Quick baseline |
| **Cross-Attn** | Rich interaction, interpretable | More params | $O(N \cdot M \cdot d)$ per layer | Most tasks (SOTA) |
| **Gated** | Adaptive, lightweight | Simple weighting | $O(d)$ — negligible overhead | When modalities vary in quality |
| **Bilinear** | Captures multiplicative interactions | Full rank is parameter-heavy | $O(d_v d_t k)$ with low-rank (MLB) | VQA, fine-grained matching |

> **Rule of thumb:** Early fusion scales poorly as sequence length grows ($N+M$ can exceed 500+ tokens in LLaVA-style models). Cross-attention gives the best interaction-to-cost ratio for vision-language tasks.

---
**Next:** Module 02 - Vision-Language Models (CLIP from scratch)